# 01 — Tái hiện baseline của tutorial

Mục tiêu: xác nhận baseline MLP 7 layer Sigmoid **không học được**, và đo định lượng
mức suy giảm gradient. Đây là mốc neo cho mọi so sánh về sau.

Kỳ vọng:
- `train_loss` đứng yên quanh `ln(10) = 2.3026` — loss của mô hình dự đoán đều 10 lớp
- `test_acc` ≈ 0.10 — mức đoán ngẫu nhiên
- gradient suy giảm theo cấp số nhân từ output về layer đầu

Nguồn baseline: *Tutorial: Vanishing Gradient*, AI VIET NAM, AI Course 2025.

## 1. Setup

In [ ]:
# Chuyển về thư mục gốc repo (notebook nằm trong notebooks/)
import os
if not os.path.exists('src') and os.path.exists('../src'):
    os.chdir('..')
print('cwd:', os.getcwd())
assert os.path.exists('src'), 'không thấy src/ — chạy notebook từ thư mục gốc repo'

In [ ]:
# Trên Colab, bỏ comment hai dòng sau:
# !git clone https://github.com/quynhanhnguyenthanh/Gradient-Vanishing-Solutions-.git
# %cd Gradient-Vanishing-Solutions-

import sys, os
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from config import Config
from train import run_experiment, load_data

print('GPU:', torch.cuda.is_available())

## 2. Kiểm tra dữ liệu

Khác tutorial, dataset được nạp sẵn toàn bộ lên GPU thay vì dùng `DataLoader`.
Vì không còn đi qua `transforms.ToTensor()`, phải chia 255 thủ công — quên bước này
thì input lớn gấp 255 lần và mọi kết quả sẽ sai.

In [ ]:
X_tr, y_tr, X_va, y_va, X_te, y_te = load_data()
print('device :', X_tr.device)
print('train  :', tuple(X_tr.shape), '| val:', tuple(X_va.shape), '| test:', tuple(X_te.shape))
print('range  :', X_tr.min().item(), '->', X_tr.max().item(), '(phải là 0.0 -> 1.0)')

## 3. Chạy baseline

`patience=50` để tắt early stopping, cho chạy đủ 50 epoch và quan sát toàn bộ quá trình.

In [ ]:
cfg = Config(name='nb_baseline', epochs=50, patience=50, seed=42)
res, hist, init = run_experiment(cfg)

## 4. Xác nhận baseline thất bại

In [ ]:
print(f"train_loss = {res['final_train_loss']:.4f}   (ln(10) = {np.log(10):.4f})")
print(f"test_acc   = {res['test_acc']:.4f}   (đoán ngẫu nhiên = 0.10)")
print(f"runtime    = {res['runtime_s']:.1f}s")

assert abs(res['final_train_loss'] - 2.3026) < 0.02, 'loss lệch khỏi ln(10)'
assert abs(res['test_acc'] - 0.10) < 0.02, 'accuracy lệch khỏi mức ngẫu nhiên'
print('\n>>> Baseline tái hiện ĐÚNG')

## 5. Ba đại lượng theo từng layer

- `grad_ratio` = ‖∇W‖/‖W‖ — dòng gradient, chuẩn hóa theo quy mô layer
- `act_grad`   = RMS của δ = ∂L/∂h — tín hiệu truyền ngược, phản ánh **cơ chế**
- `drift`      = ‖W_t − W_0‖/‖W_0‖ — layer nào thực sự đã học

In [ ]:
g, a, d = init['grad_ratio'], init['act_grad_rms'], hist['drift'][-1]
rows = [{'layer': k,
         'act_grad': a.get(k, float('nan')),
         'grad_ratio': g[k],
         'drift': d[k]} for k in g]
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:.3e}'))

### Đọc bảng

Cả `act_grad` và `grad_ratio` đều suy giảm với tốc độ tương đương (~7 lần mỗi layer).
Hai đại lượng nhất quán, nên với baseline này weight gradient là proxy hợp lệ cho
activation gradient — điều Glorot & Bengio cảnh báo có thể không đúng đã được kiểm chứng.

`drift` ở layer1 nhỏ hơn ở output hàng nghìn lần: layer đầu gần như **không nhúc nhích**
khỏi điểm khởi tạo sau 50 epoch.

In [ ]:
print(f"drift layer1 = {d['layer1']:.3e}")
print(f"drift output = {d['output']:.3e}")
print(f"tỉ số        = {d['output'] / d['layer1']:.0f} lần")

## 6. Mức suy giảm mỗi layer

In [ ]:
from metrics import relative_gradient, decay_slope_with_r2
from model import build_mlp

rel = init['relative_gradient']
print('Gradient co bao nhiêu lần khi truyền ngược qua mỗi layer:')
for k, v in rel.items():
    print(f'  {k:20s} {v:6.2f} lần')

m = build_mlp(cfg)
slope, r2 = decay_slope_with_r2(init['grad_ratio'], m)
print(f'\ndecay_slope = {slope:.3f} bậc/layer  ->  co {10**slope:.1f} lần mỗi layer')
print(f'R2          = {r2:.3f}  (gần 1 = suy giảm đúng theo cấp số nhân)')
print(f'tổng suy giảm toàn mạng = {slope * cfg.depth:.2f} bậc')

## 7. Hình: gradient theo layer (thang log)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ks = list(g.keys())
axes[0].plot(ks, [g[k] for k in ks], 'o-')
axes[0].set_yscale('log'); axes[0].set_ylabel(r'$\|\nabla W\|/\|W\|$')
axes[0].set_title('Weight gradient (hệ quả)')

ka = sorted(a, key=lambda s: int(s.replace('layer', '')))
axes[1].plot(ka, [a[k] for k in ka], 'o-', color='tab:orange')
axes[1].set_yscale('log'); axes[1].set_ylabel(r'RMS $\delta_l$')
axes[1].set_title('Activation gradient (cơ chế)')

for ax in axes:
    ax.tick_params(axis='x', rotation=45); ax.grid(alpha=0.3, which='both')
plt.suptitle('Trên thang log, suy giảm cấp số nhân thành đường thẳng')
plt.tight_layout(); plt.show()

## 8. Heatmap: vanishing có phải hiện tượng nhất thời không?

In [ ]:
hists = [h for h in hist['grad_ratio'] if h]
layers = list(hists[0].keys())
mat = np.array([[ep[l] for ep in hists] for l in layers])

plt.figure(figsize=(10, 4.5))
plt.imshow(np.log10(mat + 1e-12), aspect='auto', cmap='viridis')
plt.colorbar(label=r'$\log_{10}(\|\nabla W\|/\|W\|)$')
plt.yticks(range(len(layers)), layers)
plt.xlabel('Điểm đo'); plt.ylabel('Layer (input -> output)')
plt.title('Các layer đầu tối suốt quá trình train')
plt.tight_layout(); plt.show()

## 9. Slope có biến đổi theo thời gian không?

Đây là câu hỏi TA đặt ra. Nếu slope không đổi, có **hai** cách giải thích:
vanishing là trạng thái ổn định, hoặc model không học gì nên trọng số không đổi.
Chỉ `drift` ở mục 5 mới phân biệt được — và nó cho thấy cách giải thích thứ hai đúng.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(hist['slope'], 'o-', label='decay_slope')
plt.plot(hist['r2'], 's-', label=r'$R^2$')
plt.xlabel('Điểm đo'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Baseline: slope phẳng, R2 luôn ~0.99 — hoàn toàn tĩnh')
plt.tight_layout(); plt.show()

print(f"slope: {hist['slope'][0]:.3f} -> {hist['slope'][-1]:.3f}")
print(f"R2   : {hist['r2'][0]:.3f} -> {hist['r2'][-1]:.3f}")

---

## Kết luận

Baseline tái hiện đúng tutorial. Ba điều đo được mà tutorial không có:

1. `decay_slope ≈ 0.92` bậc/layer, tức gradient co ~8.3 lần mỗi khi qua một layer Sigmoid
2. `R² ≈ 0.99` xác nhận suy giảm đúng theo cấp số nhân, nên slope là chỉ số hợp lệ
3. `drift` ở layer1 nhỏ hơn output hàng nghìn lần — các layer đầu bị đóng băng ở trạng thái khởi tạo

Điều 3 quan trọng: nó cho thấy việc slope không đổi theo thời gian **không** phải bằng chứng
cho tính ổn định của hiện tượng, mà chỉ là hệ quả của việc trọng số không được cập nhật.

Bước tiếp theo: `02_analysis.ipynb` phân tích toàn bộ 216 run.